# 04 - Model Evaluation & Backtesting

Deep-dive evaluation of ForecastFlow models including walk-forward backtesting,
multi-model comparison, error analysis, and forecast visualization.

**Sections:**
1. Setup & Data Loading
2. Walk-Forward Backtesting
3. Model Comparison
4. Error Analysis
5. Forecast Visualization

## 1. Setup & Data Loading

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import yaml

from src.evaluation.metrics import (
    calculate_rmse,
    calculate_mae,
    calculate_mape,
    calculate_smape,
    calculate_all_metrics,
)
from src.evaluation.backtester import TimeSeriesBacktester
from src.models.ml_models import XGBoostForecaster, LightGBMForecaster

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100

print('Imports loaded successfully.')

In [ ]:
# Load project configuration
with open('../config/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

print('Config keys:', list(config.keys()))

In [ ]:
# Load featured data produced by the feature engineering pipeline
featured_data_path = '../data/processed/featured_data.parquet'
df = pd.read_parquet(featured_data_path)

print(f'Featured data shape: {df.shape}')
print(f'Date range: {df["date"].min()} to {df["date"].max()}')
df.head()

## 2. Walk-Forward Backtesting

We use an expanding-window walk-forward cross-validation strategy with 5 folds.
This simulates real-world deployment where models are retrained on all available
history and evaluated on the next unseen period.

In [ ]:
# Sample data to keep backtesting tractable
np.random.seed(42)

if 'store_id' in df.columns:
    unique_series = df['store_id'].unique()
    sample_size = min(20, len(unique_series))
    sampled_ids = np.random.choice(unique_series, size=sample_size, replace=False)
    df_sample = df[df['store_id'].isin(sampled_ids)].copy()
else:
    df_sample = df.sample(frac=0.2, random_state=42).copy()

print(f'Sampled data shape: {df_sample.shape}')

In [ ]:
# Define feature columns and target
exclude_cols = ['date', 'sales', 'target', 'store_id', 'category', 'item_id']
feature_cols = [c for c in df_sample.columns if c not in exclude_cols and df_sample[c].dtype in ['float64', 'float32', 'int64', 'int32']]

target_col = 'sales' if 'sales' in df_sample.columns else 'target'

print(f'Number of features: {len(feature_cols)}')
print(f'Target column: {target_col}')

In [ ]:
# Initialize backtester with expanding window and 5 folds
backtester = TimeSeriesBacktester(
    n_splits=5,
    strategy='expanding',
    forecast_horizon=config.get('forecast_horizon', 14),
)

# Run walk-forward CV with XGBoost
xgb_model = XGBoostForecaster(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    random_state=42,
)

cv_results = backtester.run(
    model=xgb_model,
    data=df_sample,
    feature_cols=feature_cols,
    target_col=target_col,
)

print('Walk-forward backtesting complete.')
print(f'Number of folds evaluated: {len(cv_results)}')

In [ ]:
# Summarize CV results per fold
cv_summary = pd.DataFrame(cv_results)
print('CV Results by Fold:')
cv_summary

In [ ]:
# Plot CV metrics across folds
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

metric_names = ['rmse', 'mae', 'smape']
metric_labels = ['RMSE', 'MAE', 'sMAPE (%)']

for ax, metric, label in zip(axes, metric_names, metric_labels):
    if metric in cv_summary.columns:
        folds = range(1, len(cv_summary) + 1)
        values = cv_summary[metric].values
        ax.bar(folds, values, color='steelblue', alpha=0.8)
        ax.axhline(y=np.mean(values), color='red', linestyle='--', label=f'Mean: {np.mean(values):.3f}')
        ax.set_xlabel('Fold')
        ax.set_ylabel(label)
        ax.set_title(f'{label} by CV Fold')
        ax.legend()

plt.tight_layout()
plt.suptitle('Walk-Forward CV Results (XGBoost, Expanding Window)', y=1.03, fontsize=14)
plt.show()

## 3. Model Comparison

Compare multiple model families on the held-out test set using a consistent
train/test split.

In [ ]:
# Train/test split based on time
df_sorted = df_sample.sort_values('date').reset_index(drop=True)
split_date = df_sorted['date'].quantile(0.8)

train = df_sorted[df_sorted['date'] <= split_date]
test = df_sorted[df_sorted['date'] > split_date]

X_train = train[feature_cols]
y_train = train[target_col]
X_test = test[feature_cols]
y_test = test[target_col]

print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'Split date: {split_date}')

In [ ]:
# Define models to compare
models = {
    'XGBoost': XGBoostForecaster(
        n_estimators=300, max_depth=6, learning_rate=0.05, random_state=42
    ),
    'LightGBM': LightGBMForecaster(
        n_estimators=300, max_depth=6, learning_rate=0.05, random_state=42
    ),
}

# Train each model and compute test-set metrics
comparison_rows = []

for name, model in models.items():
    print(f'Training {name}...')
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    metrics = calculate_all_metrics(y_test.values, preds)
    metrics['model'] = name
    comparison_rows.append(metrics)
    print(f'  {name} -- RMSE: {metrics["rmse"]:.4f}, MAE: {metrics["mae"]:.4f}, sMAPE: {metrics["smape"]:.2f}%')

comparison_df = pd.DataFrame(comparison_rows).set_index('model')
print('\nModel Comparison Table:')
comparison_df

In [ ]:
# Visual comparison
fig, ax = plt.subplots(figsize=(10, 5))

plot_metrics = [c for c in ['rmse', 'mae', 'mape', 'smape'] if c in comparison_df.columns]
comparison_df[plot_metrics].plot(kind='bar', ax=ax, width=0.7)

ax.set_title('Model Comparison on Test Set')
ax.set_ylabel('Metric Value')
ax.set_xlabel('Model')
ax.legend(title='Metric', loc='upper right')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 4. Error Analysis

Investigate where and when the best model makes errors.

In [ ]:
# Use the best model (lowest RMSE) for error analysis
best_model_name = comparison_df['rmse'].idxmin()
best_model = models[best_model_name]
print(f'Best model: {best_model_name}')

# Generate predictions on test set
test_preds = best_model.predict(X_test)
test_eval = test.copy()
test_eval['prediction'] = test_preds
test_eval['residual'] = test_eval[target_col] - test_eval['prediction']
test_eval['abs_error'] = test_eval['residual'].abs()
test_eval['pct_error'] = (test_eval['abs_error'] / test_eval[target_col].clip(lower=1)) * 100

In [ ]:
# 4a. Residual distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(test_eval['residual'], bins=60, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].axvline(x=0, color='red', linestyle='--', linewidth=1.5)
axes[0].set_title('Residual Distribution')
axes[0].set_xlabel('Residual (Actual - Predicted)')
axes[0].set_ylabel('Count')

from scipy import stats
stats.probplot(test_eval['residual'].dropna(), dist='norm', plot=axes[1])
axes[1].set_title('Q-Q Plot of Residuals')

plt.tight_layout()
plt.show()

print(f'Residual mean: {test_eval["residual"].mean():.4f}')
print(f'Residual std:  {test_eval["residual"].std():.4f}')
print(f'Skewness:      {test_eval["residual"].skew():.4f}')
print(f'Kurtosis:      {test_eval["residual"].kurtosis():.4f}')

In [ ]:
# 4b. Residuals over time
daily_error = test_eval.groupby('date').agg(
    mean_residual=('residual', 'mean'),
    mean_abs_error=('abs_error', 'mean'),
).reset_index()

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].plot(daily_error['date'], daily_error['mean_residual'], color='steelblue', linewidth=1)
axes[0].axhline(y=0, color='red', linestyle='--', alpha=0.7)
axes[0].fill_between(daily_error['date'], daily_error['mean_residual'], 0, alpha=0.15, color='steelblue')
axes[0].set_title('Mean Residual Over Time')
axes[0].set_ylabel('Mean Residual')

axes[1].plot(daily_error['date'], daily_error['mean_abs_error'], color='darkorange', linewidth=1)
axes[1].set_title('Mean Absolute Error Over Time')
axes[1].set_ylabel('MAE')
axes[1].set_xlabel('Date')

plt.tight_layout()
plt.show()

In [ ]:
# 4c. Residuals by day-of-week
test_eval['day_of_week'] = pd.to_datetime(test_eval['date']).dt.day_name()
dow_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.boxplot(
    data=test_eval, x='day_of_week', y='residual',
    order=dow_order, ax=axes[0], palette='Set2'
)
axes[0].set_title('Residuals by Day of Week')
axes[0].set_xlabel('Day of Week')
axes[0].set_ylabel('Residual')
axes[0].tick_params(axis='x', rotation=45)

# Residuals by store (top 10 by MAE)
if 'store_id' in test_eval.columns:
    store_mae = test_eval.groupby('store_id')['abs_error'].mean().sort_values(ascending=False)
    top_stores = store_mae.head(10).index
    sns.boxplot(
        data=test_eval[test_eval['store_id'].isin(top_stores)],
        x='store_id', y='residual', ax=axes[1], palette='Set3'
    )
    axes[1].set_title('Residuals by Store (Top 10 Worst MAE)')
    axes[1].set_xlabel('Store ID')
    axes[1].set_ylabel('Residual')
    axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# 4d. Residuals by category
if 'category' in test_eval.columns:
    fig, ax = plt.subplots(figsize=(12, 5))
    sns.boxplot(data=test_eval, x='category', y='residual', ax=ax, palette='husl')
    ax.set_title('Residuals by Category')
    ax.set_xlabel('Category')
    ax.set_ylabel('Residual')
    ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
# 4e. Identify worst-performing series
group_cols = [c for c in ['store_id', 'category', 'item_id'] if c in test_eval.columns]

if group_cols:
    series_perf = test_eval.groupby(group_cols).agg(
        mean_abs_error=('abs_error', 'mean'),
        median_abs_error=('abs_error', 'median'),
        mean_pct_error=('pct_error', 'mean'),
        n_obs=('residual', 'count'),
    ).sort_values('mean_abs_error', ascending=False).reset_index()

    print('Top 15 Worst-Performing Series (by MAE):')
    display(series_perf.head(15))

    print('\nTop 15 Best-Performing Series (by MAE):')
    display(series_perf.tail(15))
else:
    print('No grouping columns found for per-series analysis.')

## 5. Forecast Visualization

Plot actual vs. predicted for the best, worst, and median series,
including approximate confidence intervals.

In [ ]:
def plot_forecast_with_ci(series_df, title, target_col, residual_std):
    """Plot forecast with 95% confidence interval for a single series."""
    fig, ax = plt.subplots(figsize=(14, 5))

    series_sorted = series_df.sort_values('date')
    dates = series_sorted['date']
    actual = series_sorted[target_col]
    predicted = series_sorted['prediction']

    # 95% CI based on residual std
    ci_upper = predicted + 1.96 * residual_std
    ci_lower = predicted - 1.96 * residual_std

    ax.plot(dates, actual, label='Actual', color='black', linewidth=1.5)
    ax.plot(dates, predicted, label='Predicted', color='steelblue', linewidth=1.5, linestyle='--')
    ax.fill_between(dates, ci_lower, ci_upper, alpha=0.15, color='steelblue', label='95% CI')

    mae = (actual - predicted).abs().mean()
    ax.set_title(f'{title}  |  MAE: {mae:.2f}')
    ax.set_xlabel('Date')
    ax.set_ylabel(target_col.title())
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
if group_cols:
    # Residual std for CI estimation
    global_residual_std = test_eval['residual'].std()

    # Worst series
    worst_key = tuple(series_perf.iloc[0][group_cols].values)
    worst_mask = (test_eval[group_cols] == series_perf.iloc[0][group_cols]).all(axis=1)
    plot_forecast_with_ci(
        test_eval[worst_mask], f'Worst Series: {worst_key}', target_col, global_residual_std
    )

    # Best series
    best_key = tuple(series_perf.iloc[-1][group_cols].values)
    best_mask = (test_eval[group_cols] == series_perf.iloc[-1][group_cols]).all(axis=1)
    plot_forecast_with_ci(
        test_eval[best_mask], f'Best Series: {best_key}', target_col, global_residual_std
    )

    # Median series
    median_idx = len(series_perf) // 2
    median_key = tuple(series_perf.iloc[median_idx][group_cols].values)
    median_mask = (test_eval[group_cols] == series_perf.iloc[median_idx][group_cols]).all(axis=1)
    plot_forecast_with_ci(
        test_eval[median_mask], f'Median Series: {median_key}', target_col, global_residual_std
    )
else:
    # Single series fallback
    global_residual_std = test_eval['residual'].std()
    plot_forecast_with_ci(test_eval, 'Overall Forecast', target_col, global_residual_std)

---

**Summary:**
- Walk-forward backtesting with 5 expanding-window folds confirms model stability across time splits.
- Model comparison table highlights the best performer for production deployment.
- Error analysis reveals day-of-week patterns, store-level variance, and the hardest series to forecast.
- Confidence-interval plots give stakeholders a realistic picture of forecast uncertainty.